# Integrated Pipeline Testing

Using 4 demo documents linked at \
https://en.wikipedia.org/wiki/Fuxing_(train) \
https://en.wikipedia.org/wiki/Hexie_(train) \
https://en.wikipedia.org/wiki/MTR_CRRC_Changchun_EMU \
https://en.wikipedia.org/wiki/MTR_SP1900_EMU

In [1]:
%cd Multi-modal-RAG

/content/Multi-modal-RAG


## Imports

In [2]:
from config.settings import RAGConfig
from RAGPipeline import RAGSystem

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for th

## Initialization and Setup

With dense retrieval mode

In [3]:
config = RAGConfig(retrieval_mode="graph", collection_name="integration_test", extract_images=False, use_multimodal=False)
rag_system = RAGSystem(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


An error occurred while loading graph data from disk: [Errno 2] No such file or directory: './graph_db/integration_test/graph_data.pkl'
No existing graph found. Will build new graph when documents are ingested.


## Ingest Documents

In [4]:
rag_system.ingest_documents("./documents/")

Processed MTR CRRC Changchun EMU - Wikipedia.pdf: 6 chunks
0 images extracted
Processed Hexie (train) - Wikipedia.pdf: 25 chunks
0 images extracted
Processed Fuxing (train) - Wikipedia.pdf: 33 chunks
0 images extracted
Processed MTR SP1900 EMU - Wikipedia.pdf: 21 chunks
0 images extracted
Built knowledge graph from 85 document chunks


## Load Question Dataset

In [5]:
import json

with open("./questions_annotated.json", "r") as f:
    dataset = json.load(f)

dataset[0]

{'idx': 0,
 'question': 'What colors have been used on the livery of Fuxing trains?',
 'answers': ['Blue and white/silver (non-standard)',
  'Red/orange, brown and white/silver',
  'Red and silver',
  'Blue and purple (Asian games special livery)',
  'Green (on CR200J)']}

## Query and Generation

In [6]:
from datetime import datetime
import os
import time
from tqdm import tqdm
from typing import List, Dict


def generate_benchmark_answers_sync(
        rag_system: RAGSystem,
        dataset: List[Dict],
        output_path: str = "./results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the dataset.

    args:
    - rag_system (RAGSystem): Initialized RAGSystem instance
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []

    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")

    total = len(dataset)

    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue

            try:
                question = item.get('question')
                qid = item.get('qid', idx)
                print(f"Question: {question}")

                rag_result = rag_system.query(question, use_vlm=False)

                result = {
                    'qid': qid,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': rag_result.get('retrieval_mode'),
                        'documents_retrieved': rag_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': rag_result.get('generation_metadata', {}).get('images_used', 0),
                    }
                }

                if 'answers' in item:
                    result['reference_answers'] = item['answers']

                if "graph_metadata" in rag_result:
                    result["graph_metadata"] = rag_result["graph_metadata"]

                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')

                results.append(result)

            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")

            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)

    print(f"\nCompleted! Total processed: {len(results)}")

    return results

In [7]:
from utils.benchmark_eval_helpers import grade_with_llm_judge

GPT-4o-mini, with text-only retrieval; $k$=1

In [8]:
rag_system.config.top_k = 1

results_k1 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k1.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:06<01:00,  6.08s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:13<01:01,  6.88s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:10<00:07,  7.72s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:17<00:00,  7.01s/it]


Completed! Total processed: 11


In [9]:
grading_data_k1 = []
for result in results_k1:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }

    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k1.append(item)

grading_results_k1 = grade_with_llm_judge(
    responses=grading_data_k1,
    client=rag_system.generator.llm,
    output_file="./grading_results_k1.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k1['accuracy']:.2%}")
print(f"Correct: {grading_results_k1['correct_count']}/{grading_results_k1['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k1.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:18<00:00,  1.69s/it]


Detailed results saved to ./grading_results_k1.json

Grading Summary
Accuracy: 0.00%
Correct: 0/11

Detailed results saved to: ./grading_results_k1.json


GPT-4o-mini, with text-only retrieval; $k$=3

In [10]:
rag_system.config.top_k = 3

results_k3 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k3.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:06<01:07,  6.79s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:14<01:06,  7.38s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:14<00:07,  7.32s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:21<00:00,  7.38s/it]


Completed! Total processed: 11


In [11]:
grading_data_k3 = []
for result in results_k3:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k3.append(item)

grading_results_k3 = grade_with_llm_judge(
    responses=grading_data_k3,
    client=rag_system.generator.llm,
    output_file="./grading_results_k3.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k3['accuracy']:.2%}")
print(f"Correct: {grading_results_k3['correct_count']}/{grading_results_k3['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k3.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:20<00:00,  1.82s/it]


Detailed results saved to ./grading_results_k3.json

Grading Summary
Accuracy: 18.18%
Correct: 2/11

Detailed results saved to: ./grading_results_k3.json


GPT-4o-mini, with text-only retrieval; $k$=5

In [12]:
rag_system.config.top_k = 5

results_k5 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k5.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:08<01:21,  8.19s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:17<01:22,  9.14s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:16<00:07,  7.45s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:23<00:00,  7.62s/it]


Completed! Total processed: 11


In [13]:
grading_data_k5 = []
for result in results_k5:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k5.append(item)

grading_results_k5 = grade_with_llm_judge(
    responses=grading_data_k5,
    client=rag_system.generator.llm,
    output_file="./grading_results_k5.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k5['accuracy']:.2%}")
print(f"Correct: {grading_results_k5['correct_count']}/{grading_results_k5['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k5.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:21<00:00,  1.99s/it]


Detailed results saved to ./grading_results_k5.json

Grading Summary
Accuracy: 18.18%
Correct: 2/11

Detailed results saved to: ./grading_results_k5.json


GPT-4o-mini, with text-only retrieval; $k$=7

In [14]:
rag_system.config.top_k = 7

results_k7 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k7.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:09<01:36,  9.62s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:17<01:17,  8.59s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:16<00:07,  7.78s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:23<00:00,  7.60s/it]


Completed! Total processed: 11


In [15]:
grading_data_k7 = []
for result in results_k7:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k7.append(item)

grading_results_k7 = grade_with_llm_judge(
    responses=grading_data_k7,
    client=rag_system.generator.llm,
    output_file="./grading_results_k7.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k7['accuracy']:.2%}")
print(f"Correct: {grading_results_k7['correct_count']}/{grading_results_k7['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k7.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:18<00:00,  1.67s/it]


Detailed results saved to ./grading_results_k7.json

Grading Summary
Accuracy: 9.09%
Correct: 1/11

Detailed results saved to: ./grading_results_k7.json


GPT-4o-mini, with text-only retrieval; $k$=10

In [16]:
rag_system.config.top_k = 10

results_k10 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k10.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:09<01:31,  9.16s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:17<01:19,  8.89s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:28<00:08,  8.65s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:37<00:00,  8.82s/it]


Completed! Total processed: 11


In [17]:
grading_data_k10 = []
for result in results_k10:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k10.append(item)

grading_results_k10 = grade_with_llm_judge(
    responses=grading_data_k10,
    client=rag_system.generator.llm,
    output_file="./grading_results_k10.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k10['accuracy']:.2%}")
print(f"Correct: {grading_results_k10['correct_count']}/{grading_results_k10['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k10.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:16<00:00,  1.53s/it]


Detailed results saved to ./grading_results_k10.json

Grading Summary
Accuracy: 18.18%
Correct: 2/11

Detailed results saved to: ./grading_results_k10.json


GPT-4o-mini, with text-only retrieval; $k$=15

In [18]:
rag_system.config.top_k = 15

results_k15 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k15.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:07<01:17,  7.77s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:17<01:18,  8.74s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:21<00:08,  8.17s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:30<00:00,  8.18s/it]


Completed! Total processed: 11


In [19]:
grading_data_k15 = []
for result in results_k15:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k15.append(item)

grading_results_k15 = grade_with_llm_judge(
    responses=grading_data_k15,
    client=rag_system.generator.llm,
    output_file="./grading_results_k15.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k15['accuracy']:.2%}")
print(f"Correct: {grading_results_k15['correct_count']}/{grading_results_k15['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k15.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:16<00:00,  1.54s/it]


Detailed results saved to ./grading_results_k15.json

Grading Summary
Accuracy: 9.09%
Correct: 1/11

Detailed results saved to: ./grading_results_k15.json


GPT-4o-mini, with text-only retrieval; $k$=20

In [20]:
rag_system.config.top_k = 20

results_k20 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k20.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?


Generating answers:   9%|▉         | 1/11 [00:08<01:22,  8.21s/it]

Question: What is the CR450 and what is special about it?


Generating answers:  18%|█▊        | 2/11 [00:16<01:15,  8.41s/it]

...

Question: What quantitative differences do first-class carriages of the SP1900 train have as compared to other carriages?


Generating answers:  91%|█████████ | 10/11 [01:21<00:08,  8.25s/it]

Question: What major incidents have happened to SP1900 trains?


Generating answers: 100%|██████████| 11/11 [01:30<00:00,  8.27s/it]


Completed! Total processed: 11


In [21]:
grading_data_k20 = []
for result in results_k20:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    if "graph_metadata" in result:
        item["graph_metadata"] = result["graph_metadata"]

    grading_data_k20.append(item)

grading_results_k20 = grade_with_llm_judge(
    responses=grading_data_k20,
    client=rag_system.generator.llm,
    output_file="./grading_results_k20.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k20['accuracy']:.2%}")
print(f"Correct: {grading_results_k20['correct_count']}/{grading_results_k20['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k20.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:18<00:00,  1.70s/it]


Detailed results saved to ./grading_results_k20.json

Grading Summary
Accuracy: 9.09%
Correct: 1/11

Detailed results saved to: ./grading_results_k20.json
